# 5. Handle Class Imbalance & Model Evaluation

This notebook resamples the highly imbalanced training split using SMOTENC and runs three comparative model experiments:
- **Experiment 1 (Baseline)**: XGBoost trained on the original imbalanced training split using `scale_pos_weight=172`.
- **Experiment 2**: XGBoost trained on the SMOTENC-resampled training split using default classification threshold (0.5).
- **Experiment 3**: XGBoost trained on the SMOTENC-resampled training split with classification threshold tuning to optimize F1-score.

## 1. Imports & Setup

In [1]:
import os
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
import xgboost as xgb

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Imports successful.')

Imports successful.


## 2. Load Processed Data & Split

In [2]:
INPUT_PATH = '../../dataset/processed/credit_card_fraud_scaled.csv'

df = pd.read_csv(INPUT_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

# Drop the raw (unlogged) versions of amount / velocity_last_24h / city_population.
# Keeping both the raw and log versions of the same variable double-weights it in
# SMOTENC's Euclidean nearest-neighbor distance, and the raw versions are heavily
# outlier-dominated (see boxplots), which distorts which neighbors get selected.
RAW_DUPLICATE_COLS = ['amount', 'velocity_last_24h', 'city_population']
X = X.drop(columns=RAW_DUPLICATE_COLS)
print(f'Dropped raw duplicate columns {RAW_DUPLICATE_COLS}; X now has {X.shape[1]} columns')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train split: {X_train.shape[0]:,} rows. Test split: {X_test.shape[0]:,} rows.')
print('Train class distribution:')
print(y_train.value_counts().sort_index())

Loaded: 1,296,675 rows x 30 columns
Dropped raw duplicate columns ['amount', 'velocity_last_24h', 'city_population']; X now has 26 columns
Train split: 1,037,340 rows. Test split: 259,335 rows.
Train class distribution:
is_fraud
0    1031335
1       6005
Name: count, dtype: int64


## 2b. Held-Out Validation Split (for Threshold Tuning)

Experiment 3 tunes the decision threshold. Tuning it directly on `X_test`/`y_test` and then
reporting metrics on that same test set leaks test-set information into model selection and
inflates the reported score. Instead, we carve a validation split out of the *training* data:
SMOTENC and model training happen on the remaining training rows, the threshold is chosen on
`X_val`/`y_val`, and `X_test`/`y_test` is touched only once, at the very end, for final reporting.

In [3]:
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print(f'Train (sub) split: {X_train_sub.shape[0]:,} rows. Validation split: {X_val.shape[0]:,} rows.')
print('Train (sub) class distribution:')
print(y_train_sub.value_counts().sort_index())

Train (sub) split: 881,739 rows. Validation split: 155,601 rows.
Train (sub) class distribution:
is_fraud
0    876635
1      5104
Name: count, dtype: int64


## 3. Identify Feature Dtypes for SMOTENC

In [4]:
CONTINUOUS_COLS = ['amount_log', 'velocity_last_24h_log', 'city_population_log']
categorical_cols = [col for col in X_train_sub.columns if col not in CONTINUOUS_COLS]
categorical_features = [X_train_sub.columns.get_loc(col) for col in categorical_cols]

print(f'Continuous columns ({len(CONTINUOUS_COLS)}): {CONTINUOUS_COLS}')
print(f'Categorical columns ({len(categorical_cols)}): {categorical_cols}')

Continuous columns (3): ['amount_log', 'velocity_last_24h_log', 'city_population_log']
Categorical columns (23): ['day_of_week', 'is_weekend', 'location_mismatch', 'foreign_transaction', 'customer_age_binned', 'transaction_hour_binned', 'distance_to_merchant_binned', 'merchant_category_entertainment', 'merchant_category_food_dining', 'merchant_category_gas_transport', 'merchant_category_grocery_net', 'merchant_category_grocery_pos', 'merchant_category_health_fitness', 'merchant_category_home', 'merchant_category_kids_pets', 'merchant_category_misc_net', 'merchant_category_misc_pos', 'merchant_category_personal_care', 'merchant_category_shopping_net', 'merchant_category_shopping_pos', 'merchant_category_travel', 'gender_F', 'gender_M']


## 4. Experiment 1: Baseline XGBoost (scale_pos_weight=172)

Train an XGBoost model on the original imbalanced training dataset, using `scale_pos_weight=172` to account for the class imbalance.

In [5]:
print("Training Experiment 1 (Baseline)...")

# Compute scale_pos_weight dynamically from the actual train split rather than hardcoding it,
# so it stays correct if the split or upstream data ever changes.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Computed scale_pos_weight: {scale_pos_weight:.2f}')

clf_baseline = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_estimators=100,
    max_depth=6,
    n_jobs=-1
)
clf_baseline.fit(X_train, y_train)

y_pred_baseline = clf_baseline.predict(X_test)
y_proba_baseline = clf_baseline.predict_proba(X_test)[:, 1]

print('\n--- Experiment 1 (Baseline) Classification Report ---')
print(classification_report(y_test, y_pred_baseline))
print(f'ROC AUC Score: {roc_auc_score(y_test, y_proba_baseline):.4f}')

Training Experiment 1 (Baseline)...
Computed scale_pos_weight: 171.75

--- Experiment 1 (Baseline) Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    257834
           1       0.34      0.96      0.51      1501

    accuracy                           0.99    259335
   macro avg       0.67      0.97      0.75    259335
weighted avg       1.00      0.99      0.99    259335

ROC AUC Score: 0.9978


## 5. Apply SMOTENC to Resample Training Set

In [6]:
print("Applying SMOTENC to training dataset (using sampling_strategy=0.1 for high-speed resampling)...")
smote = SMOTENC(categorical_features=categorical_features, random_state=42, sampling_strategy=0.1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_sub, y_train_sub)
print('Resampled train distribution:')
print(pd.Series(y_train_resampled).value_counts().sort_index())

Applying SMOTENC to training dataset (using sampling_strategy=0.1 for high-speed resampling)...
Resampled train distribution:
is_fraud
0    876635
1     87663
Name: count, dtype: int64


## 6. Experiment 2: XGBoost on SMOTENC Resampled Training Data

Train an XGBoost model on the resampled training split and evaluate it on the same test split.

In [7]:
print("Training Experiment 2 (SMOTENC)...")
clf_smote = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_estimators=100,
    max_depth=6,
    n_jobs=-1
)
clf_smote.fit(X_train_resampled, y_train_resampled)

y_pred_smote = clf_smote.predict(X_test)
y_proba_smote = clf_smote.predict_proba(X_test)[:, 1]

print('\n--- Experiment 2 (SMOTENC) Classification Report ---')
print(classification_report(y_test, y_pred_smote))
print(f'ROC AUC Score: {roc_auc_score(y_test, y_proba_smote):.4f}')

Training Experiment 2 (SMOTENC)...

--- Experiment 2 (SMOTENC) Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    257834
           1       0.68      0.86      0.76      1501

    accuracy                           1.00    259335
   macro avg       0.84      0.93      0.88    259335
weighted avg       1.00      1.00      1.00    259335

ROC AUC Score: 0.9978


## 7. Experiment 3: SMOTENC + Threshold Tuning

Optimize the decision threshold on the held-out **validation** split (not the test set) to
maximize F1-score, then apply that threshold once to the test set for final reporting.

In [8]:
print("Running Experiment 3 (SMOTENC + Threshold Tuning)...")

# Get validation-set probabilities from the SMOTENC model (never touches X_test)
y_proba_val = clf_smote.predict_proba(X_val)[:, 1]

# Search for best decision threshold based on F1-score, using the validation set only
thresholds = np.arange(0.01, 1.00, 0.01)
best_threshold = 0.5
best_f1 = 0.0

for thresh in thresholds:
    y_pred_temp = (y_proba_val >= thresh).astype(int)
    f1 = f1_score(y_val, y_pred_temp)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

print(f'Optimal decision threshold (selected on validation set): {best_threshold:.2f} '
      f'with validation F1-score: {best_f1:.4f}')

# Apply the validation-selected threshold to the test set exactly once, for final reporting
y_pred_tuned = (y_proba_smote >= best_threshold).astype(int)
print(f'\n--- Experiment 3 (SMOTENC + Threshold Tuning @ {best_threshold:.2f}) Test Set Classification Report ---')
print(classification_report(y_test, y_pred_tuned))
print(f'ROC AUC Score: {roc_auc_score(y_test, y_proba_smote):.4f}')

Running Experiment 3 (SMOTENC + Threshold Tuning)...
Optimal decision threshold (selected on validation set): 0.86 with validation F1-score: 0.8057

--- Experiment 3 (SMOTENC + Threshold Tuning @ 0.86) Test Set Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    257834
           1       0.86      0.76      0.80      1501

    accuracy                           1.00    259335
   macro avg       0.93      0.88      0.90    259335
weighted avg       1.00      1.00      1.00    259335

ROC AUC Score: 0.9978


## 8. Save Outputs

Save the resampled training data (SMOTENC applied to the train-sub split, i.e. training
data minus the validation split) to `dataset/processed/credit_card_fraud_smotenc_train.csv`.

In [9]:
OUTPUT_PATH = '../../dataset/processed/credit_card_fraud_smotenc_train.csv'

train_resampled = X_train_resampled.copy()
train_resampled['is_fraud'] = y_train_resampled

output_dir = os.path.dirname(OUTPUT_PATH)
os.makedirs(output_dir, exist_ok=True)
train_resampled.to_csv(OUTPUT_PATH, index=False)

print(f'Saved resampled training data to: {OUTPUT_PATH}')
print(f'Rows: {train_resampled.shape[0]:,} | Columns: {train_resampled.shape[1]:,}')

Saved resampled training data to: ../../dataset/processed/credit_card_fraud_smotenc_train.csv
Rows: 964,298 | Columns: 27
